## Inputting all models as txt and making KDEs

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import norm, gaussian_kde

# Model definitions
MODELS = {
    "ST_PST19": dict(file="datafiles/ST_PST19", color="tab:pink",   label="Riley19: ST+PST"),
    "ST_PDT":   dict(file="datafiles/ST_PDT",   color="tab:blue",   label="Vinciguerra23: ST+PDT"),
    "ST_PST":   dict(file="datafiles/ST_PST",   color="tab:orange", label="Vinciguerra23: ST+PST"),
    "PDT_U":    dict(file="datafiles/PDT_U",    color="tab:green",  label="Vinciguerra23: PDT--U"),
    "ST_U":     dict(file="datafiles/ST_U",     color="tab:cyan",   label="Vinciguerra23: ST--U"),
    "2spot":    dict(file="datafiles/2spot",    color="tab:red",    label="Miller19: 2spot"),
    "3spot":    dict(file="datafiles/3spot",    color="tab:purple", label="Miller19: 3spot"),
}

# compute KDEs and individual model summary stats
datasets = {}
KDEs_2D = {}
M_med, M_sigma = {}, {}
R_med, R_sigma = {}, {}
M_min, M_max = {}, {}
R_min, R_max = {}, {}
S_Mi = {}
S_Ri = {}

for name, cfg in MODELS.items():
    data = np.loadtxt(cfg["file"])
    M, R = data[:, 0], data[:, 1]

    M_med[name], M_sigma[name] = np.median(M), np.std(M)
    R_med[name], R_sigma[name] = np.median(R), np.std(R)
    M_min[name], M_max[name] = np.min(M), np.max(M)
    R_min[name], R_max[name] = np.min(R), np.max(R)
    S_Mi[name] = max(M_med[name] - M_min[name], M_max[name] - M_med[name])
    S_Ri[name] = max(R_med[name] - R_min[name], R_max[name] - R_med[name])

    values = np.vstack([R, M])
    datasets[name] = values
    KDEs_2D[name] = gaussian_kde(values, bw_method="silverman")

# ------------------------------------------------------------
# Put KDEs on a grid
# ------------------------------------------------------------
R_grid = np.linspace(9, 17, 100)
M_grid = np.linspace(1.0, 2.3, 100)
R_mesh, M_mesh = np.meshgrid(R_grid, M_grid)
coords = np.vstack([R_mesh.ravel(), M_mesh.ravel()])

KDE_grid_values = {
    name: kde.evaluate(coords).reshape(R_mesh.shape)
    for name, kde in KDEs_2D.items()
}

In [3]:
# ------------------------------------------------------------
# 3. HDR helper + plotting helper
# ------------------------------------------------------------
def compute_hdr_levels(density, cred_levels=(0.68, 0.95)):
    vals = density.ravel()
    idx = np.argsort(vals)[::-1]
    cdf = np.cumsum(vals[idx])
    cdf /= cdf[-1]
    return [vals[idx[np.searchsorted(cdf, cl)]] for cl in cred_levels]

def plot_hdr(ax, density, color):
    lvl68, lvl95 = compute_hdr_levels(density)
    levels = [lvl95, lvl68, density.max()]

    ax.contourf(
        R_grid, M_grid, density,
        levels=levels,
        colors=[mpl.colors.to_rgba(color, 0.25),
                mpl.colors.to_rgba(color, 0.45)],
        antialiased=True
    )

    ax.contour(
        R_grid, M_grid, density,
        levels=[lvl95, lvl68],
        colors=color,
        linewidths=[1.0, 1.3],
        linestyles=["--", "-"]
    )

## Computing posterior combination

In [21]:
# Scale parameter should be set to a wide range which covers the 
# spread of the input models: more discussion of the scale parameter 
# can be found in Press 1997 paper

R_scale = 0.5 #km
M_scale = 0.1 #M_sun

# Define distributions
def f_bad_2d(R, R_mu, M, M_mu):
    return [norm.pdf(R, loc=R_mu, scale=R_scale) 
            * norm.pdf(M, loc=M_mu, scale=M_scale)]
flat_prior = np.linspace(0, 1, 100)

# Compute combination posterior on grid
posterior_2d = np.zeros(R_mesh.shape)
for i in range(R_mesh.shape[0]):
    for j in range(R_mesh.shape[1]):
        R_val, M_val = R_mesh[i, j], M_mesh[i, j]
        L_p = np.ones_like(flat_prior)
        for name in KDEs_2D:
            f_bad_val = f_bad_2d(R_val, R_mu[name], M_val, M_mu[name])
            f_good_val = KDE_grid_values[name][i, j]
            L_p *= (flat_prior * f_good_val + (1 - flat_prior) * f_bad_val) 
        posterior_2d[i, j] = np.trapz(L_p, flat_prior)
Z = np.trapz(np.trapz(posterior_2d, R_grid, axis=1), M_grid)
posterior_2d /= Z

# Convert posterior to samples for summary stats and plotting
num_samples = 40_000

R_flat = R_mesh.ravel()
M_flat = M_mesh.ravel()
posterior_flat = posterior_2d.ravel()
weights = posterior_flat / posterior_flat.sum()

idx = np.random.choice(len(weights), size=num_samples, p=weights)
sampled_R = R_flat[idx]
sampled_M = M_flat[idx]

samples = np.column_stack([sampled_R, sampled_M])
posterior_file = "combined_posterior_samples.txt"
np.savetxt(
    posterior_file,
    samples,
    fmt="%.6f",
    header="Radius    Mass"
)

print("Saved posterior samples to 'combined_posterior_samples.txt'")

# Summary statistics
def summarize(samples):
    med = np.median(samples)
    lo = med - np.percentile(samples, 16)
    hi = np.percentile(samples, 84) - med
    return med, lo, hi

R_med, R_lo, R_hi = summarize(sampled_R)
M_med, M_lo, M_hi = summarize(sampled_M)

combo_R_sigma = np.std(sampled_R)
combo_M_sigma = np.std(sampled_M)

R_mu["posterior"] = np.mean(sampled_R)
M_mu["posterior"] = np.mean(sampled_M)

print(rf"Combined Posterior best estimate for R: ${R_med:.4f}^{{+{R_hi:.4f}}}_{{-{R_lo:.4f}}} km$")
print(rf"Combined Posterior best estimate for M: ${M_med:.4f}^{{+{M_hi:.4f}}}_{{-{M_lo:.4f}}} M_\sun$")

# Add posterior to MODELS
MODELS["posterior"] = dict(
    file=posterior_file,
    color="black",
    label="Combined Posterior",
)

# Turn posterior samples back into KDE
values = np.vstack([sampled_R, sampled_M])
KDEs_2D["posterior"] = gaussian_kde(values, bw_method="silverman")

KDE_grid_values["posterior"] = (
    KDEs_2D["posterior"]
    .evaluate(coords)
    .reshape(R_mesh.shape)
)

Saved posterior samples to 'combined_posterior_samples.txt'
Combined Posterior best estimate for R: $12.7980^{+0.4848}_{-0.4040} km$
Combined Posterior best estimate for M: $1.5253^{+0.0525}_{-0.0525} M_\sun$


## Plot combined posterior M--R contour with marginalalized distrubtions (Figure 3 Sec. 3.2)

In [4]:
import seaborn as sns

# P(R): integrate over Mass
P_R = np.trapz(posterior_2d, M_grid, axis=0)
P_R /= np.trapz(P_R, R_grid)

# P(M): integrate over Radius
P_M = np.trapz(posterior_2d, R_grid, axis=1)
P_M /= np.trapz(P_M, M_grid)

# --- Quantiles (16%, 50%, 84%) ---
def compute_quantiles(grid, pdf):
    cdf = np.cumsum(pdf)
    cdf /= cdf[-1]
    return np.interp([0.16, 0.5, 0.84], cdf, grid)

R_q16, R_q50, R_q84 = compute_quantiles(R_grid, P_R)
M_q16, M_q50, M_q84 = compute_quantiles(M_grid, P_M)

# --- JointGrid ---
plt.style.use('seaborn-v0_8-white')
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

fontsize = 27
label_fs = 20
tick_fs = 16

g = sns.JointGrid(height=8, space=0)

# Zoom window
zoom_R_min, zoom_R_max = 11.0, 16.0
zoom_M_min, zoom_M_max = 1.2, 2.0

g.ax_joint.set_xlim(zoom_R_min, zoom_R_max)
g.ax_joint.set_ylim(zoom_M_min, zoom_M_max)
g.ax_marg_x.set_xlim(zoom_R_min, zoom_R_max)
g.ax_marg_y.set_ylim(zoom_M_min, zoom_M_max)

# --- HDR contours ---
lvl68, lvl95 = compute_hdr_levels(posterior_2d, (0.68, 0.95))
levels = [lvl95, lvl68, posterior_2d.max()]

g.ax_joint.contourf(
    R_grid, M_grid, posterior_2d,
    levels=levels,
    colors=[mpl.colors.to_rgba('black', 0.12),
            mpl.colors.to_rgba('black', 0.28)],
    antialiased=True
)

g.ax_joint.contour(
    R_grid, M_grid, posterior_2d,
    levels=[lvl95, lvl68],
    colors='black',
    linewidths=[1.0, 1.6],
    linestyles=['--', '-']
)

# --- 68% marginal guide lines on joint panel ---
for q in (R_q16, R_q84):
    g.ax_joint.axvline(
        q,
        color='black',
        linestyle=':',
        linewidth=1.8,
        alpha=0.9,
        zorder=10
    )

for q in (M_q16, M_q84):
    g.ax_joint.axhline(
        q,
        color='black',
        linestyle=':',
        linewidth=1.8,
        alpha=0.9,
        zorder=10
    )

# Optional: median lines (solid)
g.ax_joint.axvline(R_q50, color='black', linestyle='-.', linewidth=2.2, zorder=10)
g.ax_joint.axhline(M_q50, color='black', linestyle='-.', linewidth=2.2, zorder=10)

# --- Marginals ---
g.ax_marg_x.plot(R_grid, P_R, color='black', lw=2)
g.ax_marg_x.fill_between(R_grid, P_R, color='black', alpha=0.25)

g.ax_marg_y.plot(P_M, M_grid, color='black', lw=2)
g.ax_marg_y.fill_betweenx(M_grid, P_M, color='black', alpha=0.25)

# --- 68% CI dotted lines ---
for q in (R_q16, R_q84):
    g.ax_marg_x.axvline(q, color='black', linestyle=':', linewidth=1.8)

for q in (M_q16, M_q84):
    g.ax_marg_y.axhline(q, color='black', linestyle=':', linewidth=1.8)

# Optional: median lines
g.ax_marg_x.axvline(R_q50, color='black', linestyle='-.', linewidth=2.2)
g.ax_marg_y.axhline(M_q50, color='black', linestyle='-.', linewidth=2.2)

# --- Labels & ticks ---
g.ax_joint.set_xlabel("Radius (km)", fontsize=label_fs)
g.ax_joint.set_ylabel("Mass ($M_\\odot$)", fontsize=label_fs)

g.ax_joint.tick_params(labelsize=tick_fs, which='both', top=True, right=True)
g.ax_marg_x.tick_params(labelsize=tick_fs - 2)
g.ax_marg_y.tick_params(labelsize=tick_fs - 2)

# --- Title ---
g.fig.suptitle(
    "Combined Posterior with Marginal Distributions",
    fontsize=fontsize,
    y=1.05
)

# --- Save & show ---
g.fig.savefig(
    "figures/MarginalPosteriors.png",
    dpi=300
)

plt.show()

NameError: name 'posterior_2d' is not defined

## Plot the combined posterior against the individual models' M--R contours (Figure 2, Sec 3.2)

In [ ]:
plt.style.use('seaborn-v0_8-white')
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

# Posterior levels
posterior = KDE_grid_values['posterior']
posterior_lvl68, posterior_lvl95 = compute_hdr_levels(posterior, (0.68, 0.95))

# List of models compared with posterior
compare_series = [
    (key, MODELS[key]["color"], MODELS[key]["label"])
    for key in MODELS
]

# 2 × 4 layout
fig, axes = plt.subplots(2, 4, figsize=(24, 12), constrained_layout=True)

def rgba(color, alpha):
    return mpl.colors.to_rgba(color, alpha)

# Panels 1–7
for ax, (key, color, label) in zip(axes.flatten()[:7], compare_series):

    density = KDE_grid_values[key]
    lvl68, lvl95 = compute_hdr_levels(density, (0.68, 0.95))

    # Fills
    ax.contourf(
        R_grid, M_grid, posterior,
        levels=[posterior_lvl95, posterior_lvl68, posterior.max()],
        colors=[rgba("black", 0.10), rgba("black", 0.25)],
        antialiased=True
    )
    ax.contourf(
        R_grid, M_grid, density,
        levels=[lvl95, lvl68, density.max()],
        colors=[rgba(color, 0.20), rgba(color, 0.45)],
        antialiased=True
    )

    # Outlines
    ax.contour(
        R_grid, M_grid, posterior,
        levels=[posterior_lvl95, posterior_lvl68],
        colors='black',
        linewidths=[1.8, 2.5],
        linestyles=['--', '-']
    )

    ax.contour(
        R_grid, M_grid, density,
        levels=[lvl95, lvl68],
        colors=color,
        linewidths=[1.3, 1.8],
        linestyles=['--', '-']
    )

    ax.set_title(label, fontsize=32)
    ax.set_xlabel("Radius (km)", fontsize=28)
    ax.set_ylabel("Mass ($M_\\odot$)", fontsize=28)
    ax.minorticks_on()
    ax.tick_params(labelsize=12)

    # Legends
    ax.legend(
        handles=[
            mpl.lines.Line2D([], [], color=color, linewidth=2, label=label),
            mpl.lines.Line2D([], [], color='black', linewidth=2, label='Posterior')
        ],
        fontsize=18,
        loc='upper left',
        frameon=True,
        fancybox=True,
        framealpha=0.8
    )

# Panel 8: all models
ax_all = axes.flatten()[7]

# Individual models
for key, color, label in compare_series:
    density = KDE_grid_values[key]
    lvl68, lvl95 = compute_hdr_levels(density, (0.68, 0.95))

    ax_all.contourf(
        R_grid, M_grid, density,
        levels=[lvl95, lvl68, density.max()],
        colors=[rgba(color, 0.05), rgba(color, 0.12)],
        antialiased=True
    )

    ax_all.contour(
        R_grid, M_grid, density,
        levels=[lvl95, lvl68],
        colors=color,
        linewidths=[0.8, 1.1],
        linestyles=['--', '-']
    )

# Posterior
ax_all.contourf(
    R_grid, M_grid, posterior,
    levels=[posterior_lvl95, posterior_lvl68, posterior.max()],
    colors=[rgba("black", 0.10), rgba("black", 0.25)],
    antialiased=True
)

ax_all.contour(
    R_grid, M_grid, posterior,
    levels=[posterior_lvl95, posterior_lvl68],
    colors='black',
    linewidths=[1.8, 2.5],
    linestyles=['--', '-']
)

ax_all.set_title("All Models", fontsize=32)
ax_all.set_xlabel("Radius (km)", fontsize=28)
ax_all.set_ylabel("Mass ($M_\\odot$)", fontsize=28)
ax_all.minorticks_on()
ax_all.tick_params(labelsize=12)

fig.suptitle(
    "Model M--R Posteriors Compared to Bayesian Combination",
    fontsize=48
)

fig.savefig(
    "figures/2x4PosteriorvsIndividual_withLegends.png",
    dpi=300
)

plt.show()

NameError: name 'compute_hdr_levels' is not defined

## Statistical Credibility Scoring (Table 2, Sec. 3.3)

In [ ]:
# Score each individual measurement against the posterior
P_vi_1 = {}
epsilon = 1e-300
for name in KDEs_2D:
    prob_grid = np.zeros(R_mesh.shape)
    for i in range(R_mesh.shape[0]):
        for j in range(R_mesh.shape[1]):
            R_val, M_val = R_mesh[i, j], M_mesh[i, j]
            f_good = KDE_grid_values[name][i, j]
            f_bad = f_bad_2d(R_val, R_mu[name], M_val, M_mu[name])
            p_term = (flat_prior * f_good) / (flat_prior * f_good + (1 - flat_prior) * f_bad + epsilon)
            prob_grid[i, j] = np.trapz(p_term, flat_prior) * posterior_2d[i, j]
    P_vi_1[name] = np.trapz(np.trapz(prob_grid, R_grid, axis=1), M_grid)
    print(f"P_i for {name}: {P_vi_1[name]:.4f}")

P_i for ST_PST19: 0.7574
P_i for ST_PDT: 0.8220
P_i for ST_PST: 0.4480
P_i for PDT_U: 0.8107
P_i for ST_U: 0.7216
P_i for 2spot: 0.8238
P_i for 3spot: 0.8877
P_i for posterior: 0.9395


## Scale testing

In [ ]:
# ============================================================
# Fractional sensitivity of posterior to S_R and S_M
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Scale ranges (separate and physically motivated)
# ------------------------------------------------------------
n_scales = 26
R_scales = np.linspace(1.0, 20.0, n_scales)
M_scales = np.linspace(0.1, 2.0, n_scales)

# Fiducial values (middle of each range)
R_scale_fid = np.median(R_scales)
M_scale_fid = np.median(M_scales)


# ------------------------------------------------------------
# Helper: fractional change relative to fiducial
# ------------------------------------------------------------
def frac_change(x, x0):
    return (np.array(x) - x0) / x0


# ------------------------------------------------------------
# Fiducial posterior + summary
# ------------------------------------------------------------
posterior_fid = compute_posterior_2d(R_scale_fid, M_scale_fid)
fid_stats = summarize_posterior(posterior_fid)

R_med_0  = fid_stats["R_med"]
R_std_0  = fid_stats["R_std"]
M_med_0  = fid_stats["M_med"]
M_std_0  = fid_stats["M_std"]


# ============================================================
# Sweep S_R (fix S_M = fiducial)
# ============================================================
R_med_R, R_std_R, M_med_R, M_std_R = [], [], [], []

for SR in R_scales:
    post = compute_posterior_2d(SR, M_scale_fid)
    stats = summarize_posterior(post)

    R_med_R.append(stats["R_med"])
    R_std_R.append(stats["R_std"])
    M_med_R.append(stats["M_med"])
    M_std_R.append(stats["M_std"])


# ============================================================
# Sweep S_M (fix S_R = fiducial)
# ============================================================
R_med_M, R_std_M, M_med_M, M_std_M = [], [], [], []

for SM in M_scales:
    post = compute_posterior_2d(R_scale_fid, SM)
    stats = summarize_posterior(post)

    R_med_M.append(stats["R_med"])
    R_std_M.append(stats["R_std"])
    M_med_M.append(stats["M_med"])
    M_std_M.append(stats["M_std"])


# ============================================================
# Plot: fractional change vs S_R
# ============================================================
fig, ax = plt.subplots(2, 1, figsize=(8, 10), sharex=True)

ax[0].plot(R_scales, frac_change(R_med_R, R_med_0),
           label=r"$\Delta R_{\rm med}/R_{\rm med}$")
ax[0].plot(R_scales, frac_change(R_std_R, R_std_0),
           label=r"$\Delta\sigma_R/\sigma_R$")
ax[0].axhline(0, color="k", ls=":")
ax[0].set_ylabel("Fractional change")
ax[0].legend()

ax[1].plot(R_scales, frac_change(M_med_R, M_med_0),
           label=r"$\Delta M_{\rm med}/M_{\rm med}$")
ax[1].plot(R_scales, frac_change(M_std_R, M_std_0),
           label=r"$\Delta\sigma_M/\sigma_M$")
ax[1].axhline(0, color="k", ls=":")
ax[1].set_xlabel(r"$S_R$")
ax[1].set_ylabel("Fractional change")
ax[1].legend()

fig.suptitle("Fractional Sensitivity to Radius Scale $S_R$")
plt.tight_layout()
plt.show()


# Plot: fractional change vs S_M
fig, ax = plt.subplots(2, 1, figsize=(8, 10), sharex=True)

ax[0].plot(M_scales, frac_change(R_med_M, R_med_0),
           label=r"$\Delta R_{\rm med}/R_{\rm med}$")
ax[0].plot(M_scales, frac_change(R_std_M, R_std_0),
           label=r"$\Delta\sigma_R/\sigma_R$")
ax[0].axhline(0, color="k", ls=":")
ax[0].set_ylabel("Fractional change")
ax[0].legend()

ax[1].plot(M_scales, frac_change(M_med_M, M_med_0),
           label=r"$\Delta M_{\rm med}/M_{\rm med}$")
ax[1].plot(M_scales, frac_change(M_std_M, M_std_0),
           label=r"$\Delta\sigma_M/\sigma_M$")
ax[1].axhline(0, color="k", ls=":")
ax[1].set_xlabel(r"$S_M$")
ax[1].set_ylabel("Fractional change")
ax[1].legend()

fig.suptitle("Fractional Sensitivity to Mass Scale $S_M$")
plt.tight_layout()
plt.show()